In [1]:
## 14.3.2025

Part 2 - 

In [1]:
import numpy as np
import pandas as pd
import os
import json
import re
import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.extraction import ResultsReducer
from tqdm import tqdm
from dotenv import load_dotenv

# -----------------------------
# 1. Load OpenAI API Key
# -----------------------------
env_path = "/Users/dromar/tinytroupe/.env.local"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ ERROR: OpenAI API Key is missing! Make sure .env.local is set.")
    sys.exit(1)


# Create an explicit directory for saving interaction logs
output_dir = "output_conversations_immigration_expandedPersona_17032025"
os.makedirs(output_dir, exist_ok=True)  # Creates dir if not existing


Looking for default config on: /opt/anaconda3/envs/tinytroupe/lib/python3.10/site-packages/tinytroupe/utils/../config.ini
Failed to find custom config on: /Users/dromar/Documents/MyDrive/Research/Persuasion Agents/Data/Scripts/Amsalem2019/config.ini
Will use only default values. IF THINGS FAIL, TRY CUSTOMIZING MODEL, API TYPE, etc.

!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inacurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!


Current TinyTroupe configuration 
[OpenAI]
api_type = openai
azure_api_version = 2023-05-15
model = gpt-4o-mini
max_tokens = 4000
temperature = 1.2
freq_penalty = 0.0
presence_penalty = 0.0
timeout = 60
max_attempts = 5
waiting_time = 1
exponential_backoff_factor = 5
embedding_model = text-embedding-3-small
cache_api_calls = False
cache_file_name = openai_api_cache.pickle
max_content_d

In [2]:
# -----------------------------
# 3. Define Demographic Traits from Amsalem (2019)
# -----------------------------
np.random.seed(42)  # Ensure full reproducibility

demographics = pd.DataFrame({
    'gender': ['Female']*51 + ['Male']*49,
    'age_group': ['18-24']*13 + ['25-34']*17 + ['35-44']*17 + ['45-54']*19 + ['55-64']*17 + ['65+']*17,
    'education': ['HS_or_higher']*72 + ['Bachelor_or_higher']*28,
    'party_ID': ['Democrat']*37 + ['Republican']*26 + ['Independent']*31 + ['Other']*6
})

# Shuffle and reset index
demographics = demographics.sample(frac=1, random_state=42).reset_index(drop=True)

# Add explicitly measured traits
demographics['issue_salience'] = np.clip(np.random.normal(loc=3.71, scale=1.3, size=100), 1, 5)
demographics['political_interest'] = np.clip(np.random.normal(loc=6.81, scale=2.72, size=100), 0, 10)
demographics['need_to_evaluate'] = np.clip(np.random.normal(loc=3.25, scale=1.0, size=100), 1, 5)
demographics['political_knowledge'] = np.random.choice(['low', 'medium', 'high'], size=100, p=[0.3, 0.4, 0.3])


In [3]:
# -----------------------------
# 4. Inspecting the Created Agent Traits
# -----------------------------

# Print distributions to ensure accuracy
print("Gender Distribution:\n", demographics['gender'].value_counts(normalize=True))
print("\nAge Distribution:\n", demographics['age_group'].value_counts(normalize=True))
print("\nParty ID Distribution:\n", demographics['party_ID'].value_counts(normalize=True))

# Print summary statistics of numerical variables
print("\nIssue Salience:\n", demographics['issue_salience'].describe())
print("\nPolitical Interest:\n", demographics['political_interest'].describe())
print("\nNeed to Evaluate:\n", demographics['need_to_evaluate'].describe())
print("\nPolitical Knowledge Distribution:\n", demographics['political_knowledge'].value_counts(normalize=True))

Gender Distribution:
 gender
Female    0.51
Male      0.49
Name: proportion, dtype: float64

Age Distribution:
 age_group
45-54    0.19
65+      0.17
55-64    0.17
35-44    0.17
25-34    0.17
18-24    0.13
Name: proportion, dtype: float64

Party ID Distribution:
 party_ID
Democrat       0.37
Independent    0.31
Republican     0.26
Other          0.06
Name: proportion, dtype: float64

Issue Salience:
 count    100.000000
mean       3.523024
std        1.065928
min        1.000000
25%        2.928823
50%        3.544957
75%        4.237738
max        5.000000
Name: issue_salience, dtype: float64

Political Interest:
 count    100.000000
mean       6.680049
std        2.229632
min        1.590942
25%        4.618603
50%        7.038772
75%        8.273824
max       10.000000
Name: political_interest, dtype: float64

Need to Evaluate:
 count    100.000000
mean       3.286489
std        0.967362
min        1.000000
25%        2.594556
50%        3.347696
75%        3.954437
max        5.000

In [ ]:
import time
import logging
from tqdm import tqdm

# Logging setup
logging.basicConfig(filename="agent_generation_errors.log", level=logging.ERROR)

context_text = "You are generating politically diverse American adult participants for a study on political persuasion. Each participant should have a unique, realistic, and internally consistent political profile based on traits from U.S. demographic patterns. Be concise but informative."
factory = TinyPersonFactory(context_text=context_text)

def adaptive_delay(base_delay=2, max_delay=20, failure_count=0):
    """Implements exponential backoff for API requests, maxing at 20s."""
    return min(base_delay * (2 ** failure_count), max_delay)

def create_agents(df, factory, base_delay=2):
    """
    Generates TinyTroupe agents based on demographic traits.
    Implements adaptive delay (exponential backoff) to reduce API overload.
    """
    agents = []
    failed_agents = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Creating Agents"):
        description = (
            f"Gender: {row['gender']}. Age: {row['age_group']}. Education: {row['education']}. "
            f"Party ID: {row['party_ID']}. Political Interest: {row['political_interest']:.1f}/10. "
            f"Issue Salience: {row['issue_salience']:.1f}/5. Political Knowledge: {row['political_knowledge']}. "
            f"Need to Evaluate: {row['need_to_evaluate']:.1f}/5."
        )

        retries = 0
        while retries < 10:  # 10 retries max
            try:
                agent = factory.generate_person(agent_particularities=description, temperature=0)
                
                if agent is None:
                    raise ValueError("Agent generation returned None")
                
                agent.name = f"Agent_{idx+1}"
                agents.append(agent)
                break  # Success → exit retry loop

            except Exception as e:
                logging.error(f"Agent {idx} failed attempt {retries+1}/10. Description: {description}. Error: {e}")
                retries += 1
                wait_time = adaptive_delay(base_delay, 20, retries)
                print(f"⚠️ Retrying agent {idx} ({retries}/10) in {round(wait_time, 2)}s...")
                time.sleep(wait_time)

        else:  # Runs if all retries fail
            print(f"❌ WARNING: Agent {idx} could not be generated after 10 attempts. Skipping.")
            failed_agents.append(idx)
    
    print(f"✅ Successfully created {len(agents)} agents out of {len(df)}.")
    print(f"❌ {len(failed_agents)} agents failed. See 'agent_generation_errors.log' for details.")
    return agents, failed_agents

# Run the function
agents, failed_agents = create_agents(demographics, factory)


Creating Agents:   0%|          | 0/100 [00:00<?, ?it/s]

Creating Agents:   1%|          | 1/100 [00:21<36:03, 21.86s/it]

In [ ]:
# # -----------------------------
# # 4. Create Agents in TinyTroupe
# # -----------------------------
# import time
# import random
# import logging

# # Logging setup for debugging
# logging.basicConfig(filename="agent_generation_errors.log", level=logging.ERROR)

# context_text = "You are creating politically diverse American adult participants for a study on political persuasion. Keep responses realistic and brief."
# factory = TinyPersonFactory(context_text=context_text)

# def create_agents(df, factory, max_retries=5, initial_delay=5, batch_size=5):
#     """
#     Generates TinyTroupe agents based on demographic traits.
#     Implements retry logic with exponential backoff to handle timeouts.
#     """
#     agents = []
#     failed_agents = []  # Store failed agent IDs for debugging

#     for idx, row in tqdm(df.iterrows(), total=len(df), desc="Creating Agents"):
#         description = (
#             f"Gender: {row['gender']}. Age: {row['age_group']}. Education: {row['education']}. "
#             f"Party ID: {row['party_ID']}. Political Interest: {row['political_interest']:.1f}/10. "
#             f"Issue Salience: {row['issue_salience']:.1f}/5. Political Knowledge: {row['political_knowledge']}. "
#             f"Need to Evaluate: {row['need_to_evaluate']:.1f}/5."
#         )

#         retries = 0
#         while retries < max_retries:
#             try:
#                 agent = factory.generate_person(agent_particularities=description, temperature=0)

#                 if agent is None:
#                     raise ValueError("Agent generation returned None")

#                 agent.name = f"Agent_{idx+1}"
#                 agents.append(agent)
#                 break  # Success, move to next agent

#             except Exception as e:
#                 logging.error(f"Agent {idx} failed: {e}")
#                 retries += 1
#                 wait_time = initial_delay * (2 ** retries) + random.uniform(0, 2)
#                 print(f"⚠️ Retrying agent {idx} ({retries}/{max_retries}) in {round(wait_time, 2)}s...")
#                 time.sleep(wait_time)

#         else:  # Runs if all retries fail
#             print(f"❌ Failed to generate agent {idx} after {max_retries} attempts.")
#             failed_agents.append(idx)

#         # Pause between batches to avoid overwhelming the API
#         if idx % batch_size == 0:
#             print("⏳ Pausing to prevent API overload...")
#             time.sleep(10)

#     print(f"✅ Successfully created {len(agents)} agents out of {len(df)}.")
#     print(f"❌ {len(failed_agents)} agents failed. See 'agent_generation_errors.log' for details.")
    
#     return agents, failed_agents

# # Run the function
# agents, failed_agents = create_agents(demographics, factory)


In [ ]:
# -----------------------------
# 5. Persuasion Experiment Function (Now Uses Random Condition Assignment)
# -----------------------------
def run_persuasion_experiment(agent, agent_id, condition):
    """
    Runs the persuasion experiment for a single agent with a randomly assigned condition.
    """
    print(f"\n▶ Running experiment for {agent.name} under condition: {condition}")

    # -----------------------------
    # Step 1: Exposure to Political Speech
    # -----------------------------
    issue = condition['issue']
    party = condition['party']
    complexity = condition['complexity']

    # Retrieve the correct quote from the candidate_quotes dictionary
    quote = candidate_quotes[issue][party][complexity]

    # Construct exposure text
    exposure_text = (
        f"Paul Miller, a {party} candidate running for the U.S. Senate, presented his perspective on {issue} "
        f"for the first time last night. In a speech at a party convention, Mr. Miller stated:\n\n"
        f'"{quote}"'
    )
    
    # Agent **listens** to the speech (without acting immediately)
    agent.listen(exposure_text)

    # -----------------------------
    # Step 2: Persuasion & Recall Questions
    # -----------------------------

    # **Revised Recall Prompt for Accuracy**
    agent.listen_and_act(f"""
    Candidate Miller just gave a speech on {issue}.
    We would like to see how much you remember by listing specific policy arguments.

    📌 **Instructions:**
    - **List all specific policy positions** you recall Candidate Miller stating.
    - **Do NOT add your own opinions, summaries, or general descriptions.**
    - **If there was only one policy position, list just that.**
    - **If you are unsure about a specific argument, do NOT guess.**

    ⏩ **Example of a Correct Response:**  
    - "Supports a path to citizenship for law-abiding illegal immigrants."  
    - "Believes naturalization will benefit the economy."  

    Now, please list the **specific policy arguments** you recall from Candidate Miller’s speech. **Please list them by number in bullet form**.
    """)

    # **Persuasion Questions**
    agent.listen_and_act(f"Please rate the extent to which you agree with Candidate Miller’s position on {issue}. 1=Strongly disagree, 2=Disagree, 3=Neither agree nor disagree, 4=Agree, 5=Strongly agree.")
    agent.listen_and_act(f"How persuasive do you find Candidate Miller’s speech? 1=Definitely not persuasive, 2=Not persuasive, 3=Neither persuasive nor unpersuasive, 4=Persuasive, 5=Definitely persuasive.")

    # -----------------------------
    # Step 3: Manipulation Checks
    # -----------------------------
    agent.listen_and_act(f"How many perspectives relevant to the issue of {issue} does Candidate Miller consider in his speech? 1=One perspective, 2=Two perspectives, 3=Three or more perspectives, 4=Don’t know.")
    agent.listen_and_act(f"Does Candidate Miller refer in his speech to some type of trade-off or compromise between conflicting perspectives on {issue}? 1=Yes, 2=No, 3=Don’t know.")
    agent.listen_and_act(f"Using the following scale, where would you place Candidate Miller's view on the issue of {issue}? 1=Very liberal, 2=Liberal, 3=Neither liberal nor conservative, 4=Conservative, 5=Very conservative.")

    # -----------------------------
    # Step 4: Extract & Save Full Interaction Log
    # -----------------------------
    reducer = ResultsReducer()

    def extract_interaction(focus_agent, source_agent, target_agent, kind, event, content, timestamp):
        """Extract structured conversation data."""
        if event == "TALK":
            author = focus_agent.name
        elif event == "CONVERSATION":
            author = "USER" if source_agent is None else source_agent.name
        else:
            return None
        return {"author": author, "message": content, "timestamp": timestamp}

    # Add reduction rules
    reducer.add_reduction_rule("TALK", extract_interaction)
    reducer.add_reduction_rule("CONVERSATION", extract_interaction)

    # Extract DataFrame
    df = reducer.reduce_agent_to_dataframe(agent, column_names=["author", "message", "timestamp"])

    # Add metadata (assigned condition details)
    df['issue'] = issue
    df['party'] = party
    df['complexity'] = complexity
    df['agent_id'] = agent.name

    # Save interaction log to CSV
    output_path = os.path.join(output_dir, f"{agent_id}.csv")
    df.to_csv(output_path, index=False)

    print(f"✅ Saved results for {agent.name} under {party} {complexity} condition to {output_path}")


In [ ]:
# Explicit list of experimental conditions based on Amsalem's design
conditions_list = [
    {'issue': 'immigration', 'party': 'Democrat', 'complexity': 'low'},
    {'issue': 'immigration', 'party': 'Democrat', 'complexity': 'moderate'},
    {'issue': 'immigration', 'party': 'Democrat', 'complexity': 'high'},
    {'issue': 'immigration', 'party': 'Republican', 'complexity': 'low'},
    {'issue': 'immigration', 'party': 'Republican', 'complexity': 'moderate'},
    {'issue': 'immigration', 'party': 'Republican', 'complexity': 'high'}]

In [ ]:
# -----------------------------
# Quotes for Exposure Treatment
# -----------------------------

# Clearly structured dictionary explicitly storing Amsalem's quotes
candidate_quotes = {
    'immigration': {
        'Democrat': {
            'low': "Solving the problem of illegal immigration to the country is a top priority for me. I favor allowing illegal immigrants who are otherwise law-abiding a path to full citizenship. The reason we need to naturalize illegal immigrants is that it will make our economy grow. I have no doubt that allowing illegal immigrants legal status is the right way to go—all other solutions to this problem just don’t make sense.",
            'moderate': "Solving the problem of illegal immigration to the country is a top priority for me. I favor allowing illegal immigrants who are otherwise law-abiding a path to full citizenship. We hear many politicians say that granting illegal immigrants legal status will encourage more people to try and come here illegally. I think these politicians are wrong. The way I see it, we need to naturalize illegal immigrants in order to make our economy grow.",
            'high': "Solving the problem of illegal immigration to the country is a top priority for me. I favor allowing illegal immigrants who are otherwise law-abiding a path to full citizenship. We need to naturalize illegal immigrants because it will make our economy grow. However, we must be careful: allowing everyone to stay may encourage more illegal immigrants to come here, and we don’t want that. My plan is to balance these two goals."
        },
        'Republican': {
            'low': "Solving the problem of illegal immigration to the country is a top priority for me. I favor increasing our law-enforcement efforts against all illegal immigrants. The reason we need to be harsher with these people is that illegal immigration burdens our economy enormously. I have no doubt that increasing law-enforcement is the right way to go. All other solutions to the problem of illegal immigration just don’t make sense.",
            'moderate': "Solving the problem of illegal immigration to the country is a top priority for me. I favor increasing our law-enforcement efforts against all illegal immigrants. We hear many politicians say that increasing law-enforcement efforts is inhumane and will tear families apart. I think these politicians are wrong. The way I see it, we need to be harsher with illegal immigrants in order to reduce the heavy burden they impose on our economy.",
            'high': "Solving the problem of illegal immigration to the country is a top priority for me. I favor increasing our law-enforcement efforts against all illegal immigrants. We need to be harsher with illegal immigrants in order to reduce the heavy burden they impose on our economy. However, we must be careful: too harsh law-enforcement efforts may easily become inhumane and tear families apart. My plan is to balance these two goals."
        }
    },
    'paid leave': {
        'Democrat': {
            'low': "The U.S. is the only industrialized country that does not guarantee paid parental leave for its entire workforce. I will introduce legislation that guarantees that all American workers who become parents are provided with 12 weeks of paid leave by their employers. There is a good reason why we need to adopt such a program: it will give working parents the time off and income they need to care for their new child.",
            'moderate': "The U.S. is the only industrialized country that does not guarantee paid parental leave for its entire workforce. I will introduce legislation that guarantees that all American workers who become parents get 12 weeks of paid leave from their employers. Some say paid leave will decrease job security since it is too expensive for small businesses, but that is incorrect. We must give working parents the time off and income they need to care for their new child.",
            'high': "The U.S. is the only industrialized country that does not guarantee paid parental leave for its entire workforce. I will introduce legislation that guarantees that all American workers who become parents get 12 weeks of paid leave from their employers. My program will balance two goals. It will give working parents the time off and income they need to care for their new child, without being too expensive for small businesses and thus decreasing job security."
        },
        'Republican': {
            'low': "Lately, some legislators have introduced legislation that offers to guarantee paid parental leave for the entire workforce. They want to oblige all American employers to provide 12 weeks of fully paid parental leave to workers who become parents. There is a good reason why I oppose the program these politicians offer. A paid leave mandate is too expensive for small businesses, which means that it will cost us plenty of jobs.",
            'moderate': "Lately, some legislators have introduced legislation offering to guarantee paid parental leave for the entire workforce. They want to oblige all American employers to provide 12 weeks of fully paid parental leave to workers who become parents. I oppose this program, which is too expensive for small businesses. Some politicians say paid leave will help workers take better care of their families, but in fact it will cost us plenty of jobs.",
            'high': "Lately, some legislators have introduced legislation offering to guarantee paid parental leave for the entire workforce. They want to oblige all American employers to provide 12 weeks of fully paid parental leave to workers who become parents. I oppose this program, but believe two goals should be balanced here. While we should avoid expensive programs that burden small businesses and will cost us plenty of jobs, we must still help workers take better care of their families."
        }
    }
}

In [ ]:
# -----------------------------
# 6. Running the Experiment for All Agents
# -----------------------------
num_agents = len(agents)

# Randomly assign conditions to each agent
assigned_conditions = np.random.choice(conditions_list, size=num_agents, replace=True)

assigned_conditions_df = pd.DataFrame(assigned_conditions)
assigned_conditions_df

NameError: name 'agents' is not defined

In [ ]:

conditions = []
# Run experiment for each agent
for agent, condition in tqdm(zip(agents, assigned_conditions), total=num_agents, desc="Running Experiment"):
    run_persuasion_experiment(agent, agent.name, condition)
    conditions.append(condition)

## save conditions to pcikle
import pickle
with open('conditions_list.pkl', 'wb') as f:
    pickle.dump(conditions, f)

NameError: name 'agents' is not defined

In [11]:
#### save the relevant files to track the results

# Save the agent demographics to a CSV file
demographics.to_csv(os.path.join(output_dir, "demographics2.csv"), index=False)
# Save the experimental conditions to a csv file
assigned_conditions = pd.DataFrame(conditions_list)
assigned_conditions.to_csv(os.path.join(output_dir, "assigned_conditions.csv"), index=False)

In [11]:
output_dir

'output_conversations_immigration_14032025'

In [13]:
assigned_conditions

,issue,party,complexity
0,immigration,Democrat,low
1,immigration,Democrat,moderate
2,immigration,Democrat,high
3,immigration,Republican,low
4,immigration,Republican,moderate
5,immigration,Republican,high
